In [ ]:
# ============================================================
# 03.0 恢复研究上下文
# 动态 CSI 800 + 月末信号日 + 无前视研究掩码
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rqdatac import (
    index_components,
    get_trading_dates,
    get_next_trading_date,
    instruments,
    is_st_stock,
    is_suspended,
)

# ---------- 1. 项目路径 ----------
PROJECT_NAME = "a-share-profitability-factor-replication"

project_candidates = [
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if path.name == PROJECT_NAME
]

if not project_candidates:
    raise RuntimeError("未找到项目根目录，请检查 Notebook 的保存位置。")

PROJECT_DIR = project_candidates[-1]
TABLE_DIR = PROJECT_DIR / "artifacts" / "tables"
META_DIR = PROJECT_DIR / "artifacts" / "metadata"
FIG_DIR = PROJECT_DIR / "artifacts" / "figures"

for directory in [TABLE_DIR, META_DIR, FIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# ---------- 2. 固定研究设定 ----------
INDEX_CODE = "000906.XSHG"       # 中证 800
START_DATE = "2016-01-04"
END_DATE = "2025-12-31"
NEW_STOCK_TRADING_DAYS = 252

# ---------- 3. 动态 CSI 800 成分券池 ----------
daily_dates = pd.DatetimeIndex(
    pd.to_datetime(
        get_trading_dates(START_DATE, END_DATE)
    )
)

component_map = index_components(
    INDEX_CODE,
    start_date=START_DATE,
    end_date=END_DATE,
)

# 变更日的真实成分；缺失值先代表“不属于该日成分”
membership_on_change_dates = pd.DataFrame({
    pd.Timestamp(date): dict.fromkeys(stocks, True)
    for date, stocks in component_map.items()
}).T

membership_on_change_dates.index = pd.to_datetime(
    membership_on_change_dates.index
)

membership_on_change_dates = (
    membership_on_change_dates
    .sort_index()
    .fillna(False)
    .astype(bool)
)

# 仅将已知变更结果向未来交易日延续
index_fix = (
    membership_on_change_dates
    .reindex(daily_dates)
    .ffill()
    .fillna(False)
    .astype(bool)
)

stock_list = index_fix.columns.tolist()

# ---------- 4. 月末信号日 ----------
signal_dates = pd.DatetimeIndex(
    pd.Series(daily_dates)
    .groupby(daily_dates.to_period("M"))
    .max()
    .tolist()
)

membership_on_signal = (
    index_fix
    .reindex(index=signal_dates, columns=stock_list)
    .fillna(False)
    .astype(bool)
)

# ---------- 5. 新股过滤：信号日 T 时已上市满 252 个交易日 ----------
listing_eligible_date = {}

for sequence, stock in enumerate(stock_list, start=1):
    listed_date = pd.Timestamp(
        instruments(stock).listed_date
    )

    listing_eligible_date[stock] = pd.Timestamp(
        get_next_trading_date(
            listed_date,
            n=NEW_STOCK_TRADING_DAYS,
        )
    )

    if sequence % 250 == 0 or sequence == len(stock_list):
        print(
            "已处理 {}/{} 只股票的上市日期".format(
                sequence,
                len(stock_list),
            )
        )

new_stock_ready = pd.DataFrame(
    {
        stock: signal_dates >= eligible_date
        for stock, eligible_date
        in listing_eligible_date.items()
    },
    index=signal_dates,
).reindex(columns=stock_list).fillna(False).astype(bool)

# ---------- 6. ST 与停牌过滤：均只使用信号日 T 状态 ----------
st_on_t = (
    is_st_stock(stock_list, START_DATE, END_DATE)
    .reindex(index=signal_dates, columns=stock_list)
    .fillna(False)
    .astype(bool)
)

suspended_on_t = (
    is_suspended(stock_list, START_DATE, END_DATE)
    .reindex(index=signal_dates, columns=stock_list)
    .fillna(False)
    .astype(bool)
)

# 不在研究阶段使用 T+1 涨停、停牌、ST 状态；
# 它们属于后续“可交易策略回测”的执行约束。
research_mask = (
    membership_on_signal
    & new_stock_ready
    & (~st_on_t)
    & (~suspended_on_t)
)

# ---------- 7. 保存可开源的汇总审计 ----------
context_summary = pd.DataFrame({
    "csi800_members": membership_on_signal.sum(axis=1),
    "after_new_stock_filter": (
        membership_on_signal & new_stock_ready
    ).sum(axis=1),
    "st_excluded": (
        membership_on_signal & st_on_t
    ).sum(axis=1),
    "suspended_excluded": (
        membership_on_signal & suspended_on_t
    ).sum(axis=1),
    "research_eligible": research_mask.sum(axis=1),
})

context_summary.index.name = "signal_date"

table_path = TABLE_DIR / "03_research_context_summary.csv"
context_summary.to_csv(
    table_path,
    encoding="utf-8-sig",
)

metadata = {
    "index_code": INDEX_CODE,
    "index_name": "CSI 800",
    "research_period": {
        "start_date": START_DATE,
        "end_date": END_DATE,
    },
    "signal_frequency": "monthly last trading day",
    "new_stock_rule": (
        "Exclude stocks that have been listed for fewer than "
        "252 trading days at signal date T."
    ),
    "st_rule": "Exclude stocks that are ST at signal date T.",
    "suspension_rule": (
        "Exclude stocks suspended at signal date T."
    ),
    "future_execution_filter": (
        "T+1 tradability, limit-up and other execution constraints "
        "are not used in research-stage factor testing."
    ),
    "raw_membership_mask_saved_to_repository": False,
}

metadata_path = META_DIR / "03_research_context_rules.json"
with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

plt.figure(figsize=(12, 5))
plt.plot(
    context_summary.index,
    context_summary["research_eligible"],
    linewidth=1.6,
    label="research_eligible",
)
plt.plot(
    context_summary.index,
    context_summary["csi800_members"],
    linewidth=1.2,
    label="csi800_members",
)
plt.title("Rebuilt CSI 800 Research Universe")
plt.xlabel("Signal Date")
plt.ylabel("Stock Count")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

figure_path = FIG_DIR / "03_research_context_coverage.png"
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

print("\n项目根目录：", PROJECT_DIR)
print("信号日期数：", len(signal_dates))
print(
    "动态 CSI 800 成分股范围：{} - {}".format(
        int(membership_on_signal.sum(axis=1).min()),
        int(membership_on_signal.sum(axis=1).max()),
    )
)
print(
    "可研究股票范围：{} - {}".format(
        int(research_mask.sum(axis=1).min()),
        int(research_mask.sum(axis=1).max()),
    )
)
print("已保存：", table_path)
print("已保存：", metadata_path)
print("已保存：", figure_path)

In [ ]:
# ============================================================
# 03.1 ROIC 与 FCFFIC：原始字段提取和可用性审计
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rqfactor import Factor, execute_factor

# ---------- 1. 前置变量检查 ----------
required_variables = [
    "stock_list",
    "signal_dates",
    "research_mask",
    "START_DATE",
    "END_DATE",
    "PROJECT_DIR",
    "TABLE_DIR",
    "META_DIR",
    "FIG_DIR",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "缺少变量：" + ", ".join(missing_variables)
    )

signal_dates = pd.DatetimeIndex(pd.to_datetime(signal_dates))

# ---------- 2. 字段定义 ----------
# ROIC：直接使用米筐点时因子
# FCFFIC：按研报思路用 FCFF / 投入资本重构
input_fields = {
    "roic_direct": "return_on_invested_capital_ttm",

    "cfo_ttm": "cash_flow_from_operating_activities_ttm_0",
    "capex_ttm": "cash_paid_for_asset_ttm_0",

    "equity_mrq": "total_equity_mrq_0",
    "interest_bearing_debt": "interest_bearing_debt",
    "cash_equivalent": "cash_equivalent",
}

# ---------- 3. 逐字段提取；仅留信号日与研究券池 ----------
factor_inputs = {}

for sequence, (input_name, field_name) in enumerate(
    input_fields.items(),
    start=1,
):
    print(
        "正在提取 {}/{}：{} ({})".format(
            sequence,
            len(input_fields),
            input_name,
            field_name,
        )
    )

    raw_panel = execute_factor(
        Factor(field_name),
        stock_list,
        START_DATE,
        END_DATE,
    )

    raw_panel.index = pd.to_datetime(raw_panel.index)

    factor_inputs[input_name] = (
        raw_panel
        .reindex(index=signal_dates, columns=stock_list)
        .astype(float)
        .where(research_mask)
    )

# ---------- 4. 为后续步骤保留内存变量 ----------
roic_direct_raw = factor_inputs["roic_direct"]

cfo_ttm_raw = factor_inputs["cfo_ttm"]
capex_ttm_raw = factor_inputs["capex_ttm"]
equity_mrq_raw = factor_inputs["equity_mrq"]
interest_bearing_debt_raw = factor_inputs[
    "interest_bearing_debt"
]
cash_equivalent_raw = factor_inputs["cash_equivalent"]

# FCFFIC 的候选分母：投入资本 = 权益 + 有息负债 - 现金
invested_capital_candidate_raw = (
    equity_mrq_raw
    + interest_bearing_debt_raw
    - cash_equivalent_raw
)

fcffic_components_ready = (
    cfo_ttm_raw.notna()
    & capex_ttm_raw.notna()
    & equity_mrq_raw.notna()
    & interest_bearing_debt_raw.notna()
    & cash_equivalent_raw.notna()
)

fcffic_positive_ic_mask = (
    fcffic_components_ready
    & (invested_capital_candidate_raw > 0)
)

# 暂按“经营现金流 - 资本开支”构造候选值；
# 下一步将依据 capex 字段符号审计决定是否确认该减法方向。
fcffic_candidate_raw = (
    (cfo_ttm_raw - capex_ttm_raw)
    / invested_capital_candidate_raw
).where(fcffic_positive_ic_mask)

# ---------- 5. 覆盖率审计 ----------
coverage = pd.DataFrame(index=signal_dates)

coverage["research_eligible"] = research_mask.sum(axis=1)

for input_name, input_panel in factor_inputs.items():
    coverage[input_name + "_valid"] = input_panel.notna().sum(axis=1)

coverage["fcffic_components_ready"] = (
    fcffic_components_ready.sum(axis=1)
)
coverage["fcffic_positive_ic"] = (
    fcffic_positive_ic_mask.sum(axis=1)
)
coverage["fcffic_candidate_valid"] = (
    fcffic_candidate_raw.notna().sum(axis=1)
)

coverage.index.name = "signal_date"

# ---------- 6. 字段符号与数量级审计 ----------
sign_records = []

for input_name, input_panel in factor_inputs.items():
    values = (
        input_panel
        .replace([np.inf, -np.inf], np.nan)
        .stack()
        .dropna()
    )

    sign_records.append({
        "input_name": input_name,
        "rqfactor_field": input_fields[input_name],
        "observations": len(values),
        "mean": values.mean(),
        "median": values.median(),
        "positive_ratio": (values > 0).mean(),
        "negative_ratio": (values < 0).mean(),
        "zero_ratio": (values == 0).mean(),
    })

sign_audit = pd.DataFrame(sign_records)

# ---------- 7. 保存可开源汇总 ----------
coverage_path = TABLE_DIR / "03_roic_fcffic_input_coverage.csv"
coverage.to_csv(
    coverage_path,
    encoding="utf-8-sig",
)

sign_path = TABLE_DIR / "03_roic_fcffic_component_signs.csv"
sign_audit.to_csv(
    sign_path,
    index=False,
    encoding="utf-8-sig",
)

metadata = {
    "roic": {
        "implementation": "Direct RQFactor field",
        "field": "return_on_invested_capital_ttm",
    },
    "fcffic_candidate": {
        "numerator_candidate": (
            "cash_flow_from_operating_activities_ttm_0 "
            "- cash_paid_for_asset_ttm_0"
        ),
        "denominator_candidate": (
            "total_equity_mrq_0 + interest_bearing_debt "
            "- cash_equivalent"
        ),
        "status": (
            "Candidate only. Confirm capex sign before finalizing "
            "the construction formula."
        ),
    },
    "raw_vendor_panels_saved_to_repository": False,
    "all_fields_aligned_to": "monthly signal date T",
}
metadata_path = META_DIR / "03_roic_fcffic_input_rules.json"
with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

# ---------- 8. 覆盖率图 ----------
plot_columns = [
    "research_eligible",
    "roic_direct_valid",
    "fcffic_components_ready",
    "fcffic_positive_ic",
    "fcffic_candidate_valid",
]

plt.figure(figsize=(12, 5))

for column in plot_columns:
    plt.plot(
        coverage.index,
        coverage[column],
        label=column,
        linewidth=1.5,
    )

plt.title("ROIC and FCFFIC Input Coverage")
plt.xlabel("Signal Date")
plt.ylabel("Stock Count")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

figure_path = FIG_DIR / "03_roic_fcffic_input_coverage.png"
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

# ---------- 9. 输出检查 ----------
print("\n===== 覆盖率摘要 =====")
print(
    coverage[
        [
            "roic_direct_valid",
            "fcffic_components_ready",
            "fcffic_positive_ic",
            "fcffic_candidate_valid",
        ]
    ].agg(["mean", "min"]).round(1)
)

print("\n===== 字段符号审计 =====")
print(
    sign_audit[
        [
            "input_name",
            "median",
            "positive_ratio",
            "negative_ratio",
        ]
    ].round(4).to_string(index=False)
)

print("\n已保存：", coverage_path)
print("已保存：", sign_path)
print("已保存：", metadata_path)
print("已保存：", figure_path)

In [ ]:
# ============================================================
# 03.2 构建 ROIC 与 FCFFIC 因子
# 行业/市值中性化 + 标准化
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import rqdatac
from rqfactor import Factor, execute_factor

# ---------- 1. 前置变量检查 ----------
required_variables = [
    "stock_list",
    "signal_dates",
    "research_mask",
    "START_DATE",
    "END_DATE",
    "roic_direct_raw",
    "fcffic_candidate_raw",
    "PROJECT_DIR",
    "TABLE_DIR",
    "META_DIR",
    "FIG_DIR",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "缺少变量：" + ", ".join(missing_variables)
    )

signal_dates = pd.DatetimeIndex(pd.to_datetime(signal_dates))

# ---------- 2. 提取当期对数市值 ----------
market_cap_panel = execute_factor(
    Factor("market_cap_3"),
    stock_list,
    START_DATE,
    END_DATE,
)

market_cap_panel.index = pd.to_datetime(market_cap_panel.index)

market_cap_panel = (
    market_cap_panel
    .reindex(index=signal_dates, columns=stock_list)
    .astype(float)
    .where(research_mask)
)

log_market_cap_raw = np.log(
    market_cap_panel.where(market_cap_panel > 0)
)

# ---------- 3. 按信号日获取历史中信 2019 一级行业 ----------
SELECTED_INDUSTRY_SOURCE = "citics_2019"
INDUSTRY_FIELD = "first_industry_name"

industry_panel_raw = pd.DataFrame(
    index=signal_dates,
    columns=stock_list,
    dtype="object",
)

for sequence, as_of_date in enumerate(signal_dates, start=1):
    industry_df = rqdatac.get_instrument_industry(
        order_book_ids=stock_list,
        source=SELECTED_INDUSTRY_SOURCE,
        level=1,
        date=pd.Timestamp(as_of_date),
        market="cn",
    ).copy()

    if "order_book_id" in industry_df.columns:
        industry_df = industry_df.set_index("order_book_id")

    industry_df.index = industry_df.index.astype(str)

    if INDUSTRY_FIELD not in industry_df.columns:
        raise ValueError(
            "未找到行业字段 {}，实际字段：{}".format(
                INDUSTRY_FIELD,
                industry_df.columns.tolist(),
            )
        )

    industry_panel_raw.loc[as_of_date] = (
        industry_df[INDUSTRY_FIELD]
        .reindex(stock_list)
        .values
    )

    if sequence % 12 == 0 or sequence == len(signal_dates):
        print(
            "已完成 {}/{} 个信号日行业查询".format(
                sequence,
                len(signal_dates),
            )
        )

industry_panel = industry_panel_raw.where(research_mask)

# ---------- 4. FCFFIC 特定行业排除 ----------
FCFFIC_EXCLUDED_INDUSTRIES = {
    "银行",
    "非银行金融",
    "综合金融",
    "房地产",
}

fcffic_industry_allowed = (
    industry_panel.notna()
    & (~industry_panel.isin(FCFFIC_EXCLUDED_INDUSTRIES))
)

roic_raw = roic_direct_raw.where(research_mask)

fcffic_raw = fcffic_candidate_raw.where(
    fcffic_industry_allowed
)

# ---------- 5. 清洗与中性化函数 ----------
MAD_N = 3.5
MIN_OBSERVATIONS = 100
MIN_INDUSTRY_GROUPS = 5


def mad_winsorize(series, n=MAD_N):
    clean = (
        series
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    median_value = clean.median()
    mad_value = (clean - median_value).abs().median()

    if len(clean) == 0 or (not np.isfinite(mad_value)) or mad_value == 0:
        return clean

    lower_bound = median_value - n * 1.4826 * mad_value
    upper_bound = median_value + n * 1.4826 * mad_value

    return clean.clip(lower=lower_bound, upper=upper_bound)


def zscore(series):
    std_value = series.std(ddof=0)

    if (not np.isfinite(std_value)) or std_value == 0:
        return pd.Series(
            np.nan,
            index=series.index,
            dtype=float,
        )

    return (series - series.mean()) / std_value


def neutralize_one_cross_section(
    raw_factor,
    log_market_cap,
    industry,
):
    data = pd.DataFrame({
        "factor": raw_factor,
        "log_market_cap": log_market_cap,
        "industry": industry,
    }).replace([np.inf, -np.inf], np.nan).dropna()

    raw_valid_count = raw_factor.notna().sum()

    if len(data) < MIN_OBSERVATIONS:
        raise ValueError(
            "有效中性化样本不足：{}".format(len(data))
        )

    if data["industry"].nunique() < MIN_INDUSTRY_GROUPS:
        raise ValueError(
            "有效行业组不足：{}".format(
                data["industry"].nunique()
            )
        )

    factor_winsorized = mad_winsorize(data["factor"])
    data = data.loc[factor_winsorized.index].copy()
    data["factor"] = factor_winsorized

    industry_dummy = pd.get_dummies(
        data["industry"],
        prefix="industry",
    ).astype(float)

    # 去掉一列虚拟变量，避免与常数项完全共线
    industry_dummy = industry_dummy.iloc[:, 1:]

    x_matrix = np.column_stack([
        np.ones(len(data)),
        data["log_market_cap"].to_numpy(dtype=float),
        industry_dummy.to_numpy(dtype=float),
    ])

    y_vector = data["factor"].to_numpy(dtype=float)

    beta, _, matrix_rank, _ = np.linalg.lstsq(
        x_matrix,
        y_vector,
        rcond=None,
    )

    residual = pd.Series(
        y_vector - x_matrix @ beta,
        index=data.index,
    )

    final_factor = zscore(residual)

    audit = {
        "raw_valid_count": int(raw_valid_count),
        "model_input_count": int(len(data)),
        "final_valid_count": int(final_factor.notna().sum()),
        "industry_groups": int(data["industry"].nunique()),
        "design_matrix_rank": int(matrix_rank),
        "market_cap_beta": float(beta[1]),
        "raw_cap_corr": float(
            data["factor"].corr(data["log_market_cap"])
        ),
        "final_cap_corr": float(
            final_factor.corr(
                data.loc[
                    final_factor.index,
                    "log_market_cap",
                ]
            )
        ),
    }

    return final_factor, audit


def construct_factor_panel(factor_name, raw_panel):
    final_panel = pd.DataFrame(
        np.nan,
        index=signal_dates,
        columns=stock_list,
        dtype=float,
    )

    audit_records = []

    for as_of_date in signal_dates:
        final_factor, audit = neutralize_one_cross_section(
            raw_factor=raw_panel.loc[as_of_date],
            log_market_cap=log_market_cap_raw.loc[as_of_date],
            industry=industry_panel.loc[as_of_date],
        )

        final_panel.loc[
            as_of_date,
            final_factor.index,
        ] = final_factor.values

        audit["signal_date"] = as_of_date
        audit["factor_name"] = factor_name
        audit_records.append(audit)

    return final_panel, pd.DataFrame(audit_records)


# ---------- 6. 构建最终因子 ----------
roic_final, roic_audit = construct_factor_panel(
    "roic",
    roic_raw,
)

fcffic_final, fcffic_audit = construct_factor_panel(
    "fcffic_reconstructed",
    fcffic_raw,
)

construction_audit = pd.concat(
    [roic_audit, fcffic_audit],
    ignore_index=True,
)

# ---------- 7. 保存可开源审计结果 ----------
audit_path = TABLE_DIR / "03_roic_fcffic_construction_audit.csv"
construction_audit.to_csv(
    audit_path,
    index=False,
    encoding="utf-8-sig",
)

metadata = {
    "roic": {
        "source_field": "return_on_invested_capital_ttm",
        "treatment": (
            "MAD winsorization, CITIC 2019 first-level industry "
            "neutralization, log-market-cap neutralization, z-score."
        ),
    },
    "fcffic_reconstructed": {
        "numerator": (
            "cash_flow_from_operating_activities_ttm_0 "
            "- cash_paid_for_asset_ttm_0"
        ),
        "denominator": (
            "total_equity_mrq_0 + interest_bearing_debt "
            "- cash_equivalent"
        ),
        "factor_specific_exclusions": sorted(
            FCFFIC_EXCLUDED_INDUSTRIES
        ),
        "treatment": (
            "MAD winsorization, CITIC 2019 first-level industry "
            "neutralization, log-market-cap neutralization, z-score."
        ),
        "implementation_note": (
            "A transparent reconstructed proxy based on available "
            "RQFactor fields; not claimed to be point-identical to "
            "the source report."
        ),
    },
    "raw_factor_values_saved_to_repository": False,
}
metadata_path = META_DIR / "03_roic_fcffic_factor_rules.json"
with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

# ---------- 8. 覆盖率图 ----------
plot_data = construction_audit.pivot(
    index="signal_date",
    columns="factor_name",
    values="final_valid_count",
)

plt.figure(figsize=(12, 5))

for factor_name in plot_data.columns:
    plt.plot(
        plot_data.index,
        plot_data[factor_name],
        label=factor_name,
        linewidth=1.6,
    )

plt.title("Final ROIC and FCFFIC Factor Coverage")
plt.xlabel("Signal Date")
plt.ylabel("Valid Stock Count")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

figure_path = FIG_DIR / "03_roic_fcffic_final_coverage.png"
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

# ---------- 9. 输出检查 ----------
fcffic_industry_excluded_count = (
    research_mask
    & industry_panel.isin(FCFFIC_EXCLUDED_INDUSTRIES)
).sum(axis=1)

summary = construction_audit.groupby(
    "factor_name"
)[
    [
        "raw_valid_count",
        "model_input_count",
        "final_valid_count",
        "industry_groups",
        "final_cap_corr",
    ]
].agg(["mean", "min"])

print("\n===== 最终因子构建摘要 =====")
print(summary.round(3))

print(
    "\nFCFFIC 每期因行业排除的平均股票数：{:.1f}".format(
        fcffic_industry_excluded_count.mean()
    )
)
print(
    "FCFFIC 每期因行业排除的最大股票数：{}".format(
        int(fcffic_industry_excluded_count.max())
    )
)

print("\n已保存：", audit_path)
print("已保存：", metadata_path)
print("已保存：", figure_path)

In [ ]:
# ============================================================
# 03.3 重建 20 日未来收益标签，并审计 ROIC / FCFFIC 配对覆盖
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rqdatac import get_price

# ---------- 1. 前置变量检查 ----------
required_variables = [
    "stock_list",
    "signal_dates",
    "roic_final",
    "fcffic_final",
    "TABLE_DIR",
    "META_DIR",
    "FIG_DIR",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "缺少变量：" + ", ".join(missing_variables)
    )

signal_dates = pd.DatetimeIndex(pd.to_datetime(signal_dates))

# ---------- 2. 读取第一阶段已确认的交易时间表 ----------
schedule_path = TABLE_DIR / "02_return_schedule.csv"

if not schedule_path.exists():
    raise FileNotFoundError(
        "找不到收益时间表：" + str(schedule_path)
    )

return_schedule = pd.read_csv(
    schedule_path,
    parse_dates=[
        "signal_date",
        "entry_date_t_plus_1_open",
        "exit_date_t_plus_21_open",
    ],
)

return_schedule = (
    return_schedule
    .set_index("signal_date")
    .reindex(signal_dates)
    .reset_index()
)

if return_schedule[
    [
        "entry_date_t_plus_1_open",
        "exit_date_t_plus_21_open",
    ]
].isna().any().any():
    raise RuntimeError("收益时间表存在缺失日期，请检查 02_return_schedule.csv。")

first_entry_date = return_schedule[
    "entry_date_t_plus_1_open"
].min()

last_exit_date = return_schedule[
    "exit_date_t_plus_21_open"
].max()

# ---------- 3. 获取复权开盘价并重建标签 ----------
open_price_long = get_price(
    order_book_ids=stock_list,
    start_date=first_entry_date,
    end_date=last_exit_date,
    frequency="1d",
    fields="open",
    adjust_type="pre",
)

open_price = (
    open_price_long["open"]
    .unstack("order_book_id")
)

open_price.index = pd.to_datetime(open_price.index)
open_price = open_price.reindex(columns=stock_list)

forward_return_20d = pd.DataFrame(
    np.nan,
    index=signal_dates,
    columns=stock_list,
    dtype=float,
)

for _, row in return_schedule.iterrows():
    signal_date = pd.Timestamp(row["signal_date"])
    entry_date = pd.Timestamp(row["entry_date_t_plus_1_open"])
    exit_date = pd.Timestamp(row["exit_date_t_plus_21_open"])

    entry_open = open_price.loc[entry_date]
    exit_open = open_price.loc[exit_date]

    valid_price = (
        entry_open.notna()
        & exit_open.notna()
        & (entry_open > 0)
        & (exit_open > 0)
    )

    forward_return_20d.loc[signal_date] = (
        exit_open / entry_open - 1
    ).where(valid_price)

# ---------- 4. ROIC / FCFFIC 配对覆盖率 ----------
factor_panels = {
    "roic": roic_final,
    "fcffic_reconstructed": fcffic_final,
}

pair_coverage_records = []

for factor_name, factor_panel in factor_panels.items():
    factor_valid = factor_panel.notna().sum(axis=1)
    return_valid = forward_return_20d.notna().sum(axis=1)

    pair_count = (
        factor_panel.notna()
        & forward_return_20d.notna()
    ).sum(axis=1)

    coverage_rate = (
        pair_count / factor_valid.replace(0, np.nan)
    )

    for as_of_date in signal_dates:
        pair_coverage_records.append({
            "signal_date": as_of_date,
            "factor_name": factor_name,
            "factor_valid": int(factor_valid.loc[as_of_date]),
            "return_valid_all_stock_list": int(
                return_valid.loc[as_of_date]
            ),
            "factor_return_paired": int(
                pair_count.loc[as_of_date]
            ),
            "pair_coverage_pct": float(
                100 * coverage_rate.loc[as_of_date]
            ),
        })

pair_coverage = pd.DataFrame(pair_coverage_records)

# ---------- 5. 保存可开源汇总 ----------
coverage_path = TABLE_DIR / "03_roic_fcffic_return_pair_coverage.csv"
pair_coverage.to_csv(
    coverage_path,
    index=False,
    encoding="utf-8-sig",
)

metadata = {
    "return_schedule_source": "02_return_schedule.csv",
    "return_definition": (
        "Entry at T+1 open; exit at T+21 open; "
        "20 trading-day holding interval."
    ),
    "price_field": "pre-adjusted open price",
    "raw_price_or_stock_level_return_saved_to_repository": False,
}
metadata_path = META_DIR / "03_roic_fcffic_return_pair_rules.json"
with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

plot_data = pair_coverage.pivot(
    index="signal_date",
    columns="factor_name",
    values="factor_return_paired",
)

plt.figure(figsize=(12, 5))

for factor_name in plot_data.columns:
    plt.plot(
        plot_data.index,
        plot_data[factor_name],
        label=factor_name,
        linewidth=1.6,
    )

plt.title("ROIC and FCFFIC Factor-Return Pair Coverage")
plt.xlabel("Signal Date")
plt.ylabel("Paired Stock Count")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

figure_path = FIG_DIR / "03_roic_fcffic_return_pair_coverage.png"
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

# ---------- 6. 输出检查 ----------
pair_summary = pair_coverage.groupby("factor_name")[
    [
        "factor_valid",
        "factor_return_paired",
        "pair_coverage_pct",
    ]
].agg(["mean", "min"])

print("\n===== 因子-收益配对摘要 =====")
print(pair_summary.round(2))

print(
    "\n收益标签区间：{} 至 {}".format(
        first_entry_date.strftime("%Y-%m-%d"),
        last_exit_date.strftime("%Y-%m-%d"),
    )
)
print("已保存：", coverage_path)
print("已保存：", metadata_path)
print("已保存：", figure_path)

In [ ]:
# ============================================================
# 03.4 ROIC 与 FCFFIC 单因子有效性检验
# IC / RankIC / 五分组 / 高低组 / 样本外表现
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t as student_t

# ---------- 1. 前置变量检查 ----------
required_variables = [
    "signal_dates",
    "roic_final",
    "fcffic_final",
    "forward_return_20d",
    "TABLE_DIR",
    "META_DIR",
    "FIG_DIR",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "缺少变量：" + ", ".join(missing_variables)
    )

signal_dates = pd.DatetimeIndex(pd.to_datetime(signal_dates))
GROUP_COUNT = 5

factor_panels = {
    "roic": roic_final,
    "fcffic_reconstructed": fcffic_final,
}

# ---------- 2. 逐期计算 IC、RankIC、五分组 ----------
metric_records = []
quintile_records = []

for factor_name, factor_panel in factor_panels.items():
    for as_of_date in signal_dates:
        data = pd.DataFrame({
            "factor": factor_panel.loc[as_of_date],
            "forward_return": forward_return_20d.loc[as_of_date],
        }).replace([np.inf, -np.inf], np.nan).dropna()

        if len(data) < 100:
            raise ValueError(
                "{} 在 {} 的样本数不足 100，只剩 {}。".format(
                    factor_name,
                    as_of_date.strftime("%Y-%m-%d"),
                    len(data),
                )
            )

        ic_value = data["factor"].corr(
            data["forward_return"],
            method="pearson",
        )

        rank_ic_value = data["factor"].corr(
            data["forward_return"],
            method="spearman",
        )

        data["quintile"] = (
            pd.qcut(
                data["factor"].rank(method="first"),
                q=GROUP_COUNT,
                labels=False,
            ) + 1
        )

        group_return = data.groupby(
            "quintile"
        )["forward_return"].mean()

        group_count = data.groupby(
            "quintile"
        )["forward_return"].count()

        metric_records.append({
            "signal_date": as_of_date,
            "factor_name": factor_name,
            "valid_stocks": len(data),
            "ic": ic_value,
            "rank_ic": rank_ic_value,
        })

        quintile_records.append({
            "signal_date": as_of_date,
            "factor_name": factor_name,
            "q1_return": group_return.get(1, np.nan),
            "q2_return": group_return.get(2, np.nan),
            "q3_return": group_return.get(3, np.nan),
            "q4_return": group_return.get(4, np.nan),
            "q5_return": group_return.get(5, np.nan),
            "high_minus_low_return": (
                group_return.get(5, np.nan)
                - group_return.get(1, np.nan)
            ),
            "q1_count": group_count.get(1, np.nan),
            "q2_count": group_count.get(2, np.nan),
            "q3_count": group_count.get(3, np.nan),
            "q4_count": group_count.get(4, np.nan),
            "q5_count": group_count.get(5, np.nan),
        })

factor_metrics = pd.DataFrame(metric_records)
factor_quintiles = pd.DataFrame(quintile_records)

# ---------- 3. 样本区间汇总函数 ----------
def t_stat_and_p_value(series):
    series = series.dropna()

    if len(series) < 2 or series.std(ddof=1) == 0:
        return np.nan, np.nan

    t_stat = (
        series.mean()
        / (series.std(ddof=1) / np.sqrt(len(series)))
    )

    p_value = 2 * student_t.sf(
        np.abs(t_stat),
        df=len(series) - 1,
    )

    return t_stat, p_value


def summarize_factor_period(
    factor_name,
    period_name,
    start_date,
    end_date,
):
    metric_data = factor_metrics[
        (factor_metrics["factor_name"] == factor_name)
        & (factor_metrics["signal_date"] >= start_date)
        & (factor_metrics["signal_date"] <= end_date)
    ]

    quintile_data = factor_quintiles[
        (factor_quintiles["factor_name"] == factor_name)
        & (factor_quintiles["signal_date"] >= start_date)
        & (factor_quintiles["signal_date"] <= end_date)
    ]

    ic_t_stat, ic_p_value = t_stat_and_p_value(
        metric_data["ic"]
    )

    rank_ic_t_stat, rank_ic_p_value = t_stat_and_p_value(
        metric_data["rank_ic"]
    )

    hml_t_stat, hml_p_value = t_stat_and_p_value(
        quintile_data["high_minus_low_return"]
    )

    return {
        "factor_name": factor_name,
        "period": period_name,
        "observations": len(metric_data),
        "average_valid_stocks": metric_data[
            "valid_stocks"
        ].mean(),
        "mean_ic": metric_data["ic"].mean(),
        "icir_annualized": (
            metric_data["ic"].mean()
            / metric_data["ic"].std(ddof=1)
            * np.sqrt(12)
        ),
        "ic_t_stat": ic_t_stat,
        "ic_p_value": ic_p_value,
        "mean_rank_ic": metric_data["rank_ic"].mean(),
        "rank_icir_annualized": (
            metric_data["rank_ic"].mean()
            / metric_data["rank_ic"].std(ddof=1)
            * np.sqrt(12)
        ),
        "rank_ic_t_stat": rank_ic_t_stat,
        "rank_ic_p_value": rank_ic_p_value,
        "positive_rank_ic_ratio": (
            metric_data["rank_ic"] > 0
        ).mean(),
        "mean_q1_return": quintile_data["q1_return"].mean(),
        "mean_q2_return": quintile_data["q2_return"].mean(),
        "mean_q3_return": quintile_data["q3_return"].mean(),
        "mean_q4_return": quintile_data["q4_return"].mean(),
        "mean_q5_return": quintile_data["q5_return"].mean(),
        "mean_high_minus_low_return": quintile_data[
            "high_minus_low_return"
        ].mean(),
        "high_minus_low_t_stat": hml_t_stat,
        "high_minus_low_p_value": hml_p_value,
        "high_minus_low_positive_ratio": (
            quintile_data["high_minus_low_return"] > 0
        ).mean(),
    }


periods = {
    "full_sample_2016_2025": (
        pd.Timestamp("2016-01-01"),
        pd.Timestamp("2025-12-31"),
    ),
    "in_sample_2016_2022": (
        pd.Timestamp("2016-01-01"),
        pd.Timestamp("2022-12-31"),
    ),
    "out_of_sample_2023_2025": (
        pd.Timestamp("2023-01-01"),
        pd.Timestamp("2025-12-31"),
    ),
}

summary_records = []

for factor_name in factor_panels:
    for period_name, (start_date, end_date) in periods.items():
        summary_records.append(
            summarize_factor_period(
                factor_name,
                period_name,
                start_date,
                end_date,
            )
        )

factor_effectiveness_summary = pd.DataFrame(summary_records)

# ---------- 4. 保存可开源聚合结果 ----------
metrics_path = TABLE_DIR / "03_roic_fcffic_ic_rankic_timeseries.csv"
factor_metrics.to_csv(
    metrics_path,
    index=False,
    encoding="utf-8-sig",
)

quintile_path = TABLE_DIR / "03_roic_fcffic_quintile_return_timeseries.csv"
factor_quintiles.to_csv(
    quintile_path,
    index=False,
    encoding="utf-8-sig",
)

summary_path = TABLE_DIR / "03_roic_fcffic_effectiveness_summary.csv"
factor_effectiveness_summary.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig",
)

metadata = {
    "factors": {
        "roic": "Direct RQFactor ROIC, cleaned and neutralized.",
        "fcffic_reconstructed": (
            "Reconstructed CFO minus capex divided by invested capital, "
            "with factor-specific financial and real-estate exclusions."
        ),
    },
    "return_label": (
        "T+1 open to T+21 open, representing a 20 trading-day holding interval."
    ),
    "grouping": (
        "Equal-count quintiles. Q1 is lowest factor, Q5 is highest factor."
    ),
    "high_minus_low": "Q5 gross return minus Q1 gross return.",
    "sample_split": {
        "in_sample": "2016-2022",
        "out_of_sample": "2023-2025",
    },
    "transaction_costs_included": False,
    "execution_constraints_included": False,
    "raw_factor_or_stock_level_return_saved_to_repository": False,
}
metadata_path = META_DIR / "03_roic_fcffic_factor_test_rules.json"
with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

# ---------- 5. 绘图 ----------
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for row_index, factor_name in enumerate(factor_panels):
    factor_metric = (
        factor_metrics[
            factor_metrics["factor_name"] == factor_name
        ]
        .set_index("signal_date")
        .sort_index()
    )

    factor_group_return = (
        factor_quintiles[
            factor_quintiles["factor_name"] == factor_name
        ]
        .set_index("signal_date")
        .sort_index()
    )

    axes[row_index, 0].plot(
        factor_metric.index,
        factor_metric["ic"],
        label="IC",
        linewidth=1.3,
    )
    axes[row_index, 0].plot(
        factor_metric.index,
        factor_metric["rank_ic"],
        label="RankIC",
        linewidth=1.3,
    )
    axes[row_index, 0].axhline(
        0,
        color="gray",
        linestyle="--",
        linewidth=1,
    )
    axes[row_index, 0].set_title(
        factor_name + ": Monthly IC and RankIC"
    )
    axes[row_index, 0].set_xlabel("Signal Date")
    axes[row_index, 0].set_ylabel("Correlation")
    axes[row_index, 0].grid(alpha=0.25)
    axes[row_index, 0].legend()

    quintile_columns = [
        "q1_return",
        "q2_return",
        "q3_return",
        "q4_return",
        "q5_return",
    ]

    quintile_nav = (
        1 + factor_group_return[quintile_columns]
    ).cumprod()

    for column in quintile_nav.columns:
        axes[row_index, 1].plot(
            quintile_nav.index,
            quintile_nav[column],
            label=column.replace("_return", "").upper(),
            linewidth=1.35,
        )

    axes[row_index, 1].set_title(
        factor_name + ": Equal-Weighted Quintile Net Value"
    )
    axes[row_index, 1].set_xlabel("Signal Date")
    axes[row_index, 1].set_ylabel("Gross Net Value")
    axes[row_index, 1].grid(alpha=0.25)
    axes[row_index, 1].legend()

plt.tight_layout()

figure_path = FIG_DIR / "03_roic_fcffic_factor_effectiveness.png"
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

# ---------- 6. 输出关键结果 ----------
display_columns = [
    "factor_name",
    "period",
    "observations",
    "mean_ic",
    "icir_annualized",
    "mean_rank_ic",
    "rank_icir_annualized",
    "positive_rank_ic_ratio",
    "mean_high_minus_low_return",
    "high_minus_low_t_stat",
]

print(
    factor_effectiveness_summary[
        display_columns
    ].round(4).to_string(index=False)
)

print("\n已保存：", metrics_path)
print("已保存：", quintile_path)
print("已保存：", summary_path)
print("已保存：", metadata_path)
print("已保存：", figure_path)

In [ ]:
# ============================================================
# 03.5 保存 ROIC 与 FCFFIC 研究决策日志
# ============================================================

from pathlib import Path
import json
import pandas as pd

# ---------- 1. 读取已经保存的汇总结果 ----------
summary_path = (
    TABLE_DIR
    / "03_roic_fcffic_effectiveness_summary.csv"
)

if not summary_path.exists():
    raise FileNotFoundError(
        "找不到有效性汇总表：" + str(summary_path)
    )

roic_fcffic_summary = pd.read_csv(summary_path)

oos_result = roic_fcffic_summary[
    roic_fcffic_summary["period"]
    == "out_of_sample_2023_2025"
].copy()

# ---------- 2. 记录研究判断，而非按结果调参 ----------
decision_map = {
    "roic": {
        "decision": "reject_as_standalone_factor",
        "reason": (
            "样本外 RankIC 为负，高低组收益也为负，"
            "说明该直接 ROIC 实现不具备稳定预测能力。"
        ),
        "next_action": (
            "仅保留为研报复现中的比较基准，不进入后续盈利能力因子库。"
        ),
    },
    "fcffic_reconstructed": {
        "decision": "candidate_not_confirmed",
        "reason": (
            "样本外 RankIC 维持正值，正 RankIC 月份占比较高；"
            "但高低组收益尚不显著，不能直接宣称为可交易 Alpha。"
        ),
        "next_action": (
            "保留为盈利能力因子库候选，并与盈利稳定性组件进行独立、"
            "预先设定规则的组合研究。"
        ),
    },
}

decision_log = oos_result.copy()

decision_log["decision"] = decision_log[
    "factor_name"
].map(lambda name: decision_map[name]["decision"])

decision_log["research_interpretation"] = decision_log[
    "factor_name"
].map(lambda name: decision_map[name]["reason"])

decision_log["next_action"] = decision_log[
    "factor_name"
].map(lambda name: decision_map[name]["next_action"])

decision_log = decision_log[
    [
        "factor_name",
        "period",
        "observations",
        "mean_rank_ic",
        "rank_icir_annualized",
        "positive_rank_ic_ratio",
        "mean_high_minus_low_return",
        "high_minus_low_t_stat",
        "high_minus_low_p_value",
        "decision",
        "research_interpretation",
        "next_action",
    ]
]

# ---------- 3. 保存 ----------
decision_path = (
    TABLE_DIR
    / "03_roic_fcffic_research_decision_log.csv"
)

decision_log.to_csv(
    decision_path,
    index=False,
    encoding="utf-8-sig",
)

metadata = {
    "out_of_sample_period": "2023-01-01 to 2025-12-31",
    "decision_principle": (
        "Do not modify factor formulas, exclusion rules, or grouping "
        "rules after observing these results."
    ),
    "research_conclusion": (
        "Direct ROIC is rejected as a standalone factor. "
        "Reconstructed FCFFIC is retained as a candidate component, "
        "not a confirmed standalone alpha."
    ),
    "next_research_stage": (
        "Create a cross-factor scorecard using all completed profitability "
        "factor evidence, then define any composite as exploratory."
    ),
}

metadata_path = (
    META_DIR
    / "03_roic_fcffic_research_decision.json"
)

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

print(
    decision_log[
        [
            "factor_name",
            "mean_rank_ic",
            "mean_high_minus_low_return",
            "high_minus_low_t_stat",
            "decision",
        ]
    ].round(4).to_string(index=False)
)

print("\n已保存：", decision_path)
print("已保存：", metadata_path)


In [ ]:
# ============================================================
# 04.1 盈利能力因子研究评分卡
# 汇总所有样本外证据、研究决策与后续用途
# ============================================================

from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

# ---------- 1. 项目路径 ----------
PROJECT_NAME = "a-share-profitability-factor-replication"

project_candidates = [
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if path.name == PROJECT_NAME
]

if not project_candidates:
    raise RuntimeError("未找到项目根目录。")

PROJECT_DIR = project_candidates[-1]
TABLE_DIR = PROJECT_DIR / "artifacts" / "tables"
META_DIR = PROJECT_DIR / "artifacts" / "metadata"
FIG_DIR = PROJECT_DIR / "artifacts" / "figures"

# ---------- 2. 读取已有研究结果 ----------
variant_summary_path = (
    TABLE_DIR
    / "02_profitability_variant_comparison.csv"
)

roic_fcffic_summary_path = (
    TABLE_DIR
    / "03_roic_fcffic_effectiveness_summary.csv"
)

variant_decision_path = (
    TABLE_DIR
    / "02_profitability_research_decision_log.csv"
)

roic_fcffic_decision_path = (
    TABLE_DIR
    / "03_roic_fcffic_research_decision_log.csv"
)

required_files = [
    variant_summary_path,
    roic_fcffic_summary_path,
    variant_decision_path,
    roic_fcffic_decision_path,
]

missing_files = [
    str(path)
    for path in required_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "缺少以下研究结果文件：\n"
        + "\n".join(missing_files)
    )

variant_summary = pd.read_csv(variant_summary_path)
roic_fcffic_summary = pd.read_csv(
    roic_fcffic_summary_path
)
variant_decision = pd.read_csv(variant_decision_path)
roic_fcffic_decision = pd.read_csv(
    roic_fcffic_decision_path
)

# ---------- 3. 只保留样本外区间 ----------
variant_oos = variant_summary[
    variant_summary["period"]
    == "out_of_sample_2023_2025"
].copy()

variant_oos = variant_oos.rename(
    columns={"factor_variant": "factor_name"}
)

roic_fcffic_oos = roic_fcffic_summary[
    roic_fcffic_summary["period"]
    == "out_of_sample_2023_2025"
].copy()

oos_summary = pd.concat(
    [variant_oos, roic_fcffic_oos],
    ignore_index=True,
    sort=False,
)

# ---------- 4. 合并已经记录的研究决策 ----------
variant_decision = variant_decision.rename(
    columns={"factor_variant": "factor_name"}
)

all_decisions = pd.concat(
    [
        variant_decision[
            [
                "factor_name",
                "decision",
                "research_interpretation",
                "next_action",
            ]
        ],
        roic_fcffic_decision[
            [
                "factor_name",
                "decision",
                "research_interpretation",
                "next_action",
            ]
        ],
    ],
    ignore_index=True,
)

scorecard = oos_summary.merge(
    all_decisions,
    on="factor_name",
    how="left",
)

# ---------- 5. 增加项目中可读的因子说明 ----------
factor_info = {
    "roe_ttm_baseline": {
        "factor_label": "原始 ROE",
        "factor_family": "ROE 对照组",
        "implementation": "RQFactor TTM ROE",
    },
    "delevered_roe": {
        "factor_label": "去杠杆 ROE",
        "factor_family": "ROE 改造",
        "implementation": (
            "ROE 对权益乘数回归后的残差"
        ),
    },
    "negative_roe_volatility": {
        "factor_label": "盈利稳定性",
        "factor_family": "ROE 改造",
        "implementation": (
            "过去 9 期 TTM ROE 波动率取负"
        ),
    },
    "stable_delevered_roe": {
        "factor_label": "稳定去杠杆 ROE",
        "factor_family": "ROE 改造",
        "implementation": (
            "去杠杆 ROE 与盈利稳定性的等权组合"
        ),
    },
    "roic": {
        "factor_label": "ROIC",
        "factor_family": "投入资本回报",
        "implementation": "直接 RQFactor ROIC",
    },
    "fcffic_reconstructed": {
        "factor_label": "重构 FCFFIC",
        "factor_family": "自由现金流回报",
        "implementation": (
            "(经营现金流 - 资本开支) / 投入资本；"
            "排除金融和房地产"
        ),
    },
}

scorecard["factor_label"] = scorecard[
    "factor_name"
].map(lambda name: factor_info[name]["factor_label"])

scorecard["factor_family"] = scorecard[
    "factor_name"
].map(lambda name: factor_info[name]["factor_family"])

scorecard["implementation"] = scorecard[
    "factor_name"
].map(lambda name: factor_info[name]["implementation"])

# 方便阅读的排序：候选组件在前，其余按 OOS RankIC 排序
decision_rank = {
    "candidate_not_confirmed": 1,
    "not_a_standalone_factor": 2,
    "reject_as_standalone_factor": 3,
}

scorecard["decision_rank"] = scorecard["decision"].map(
    decision_rank
)

scorecard = scorecard.sort_values(
    [
        "decision_rank",
        "mean_rank_ic",
    ],
    ascending=[True, False],
).reset_index(drop=True)

# ---------- 6. 保存可开源评分卡 ----------
display_columns = [
    "factor_label",
    "factor_name",
    "factor_family",
    "implementation",
    "observations",
    "mean_rank_ic",
    "rank_icir_annualized",
    "positive_rank_ic_ratio",
    "mean_high_minus_low_return",
    "high_minus_low_t_stat",
    "high_minus_low_p_value",
    "decision",
    "research_interpretation",
    "next_action",
]

scorecard = scorecard[display_columns]

scorecard_path = (
    TABLE_DIR
    / "04_profitability_factor_scorecard.csv"
)

scorecard.to_csv(
    scorecard_path,
    index=False,
    encoding="utf-8-sig",
)

metadata = {
    "purpose": (
        "Final out-of-sample scorecard for all implemented "
        "profitability factor candidates."
    ),
    "out_of_sample_period": "2023-01-01 to 2025-12-31",
    "interpretation_rule": (
        "Candidate status requires persistent positive direction but "
        "does not mean a confirmed tradable alpha."
    ),
    "portfolio_claim": (
        "No composite portfolio is selected or optimized from this "
        "scorecard, to avoid post-hoc selection bias."
    ),
    "raw_stock_level_data_saved_to_repository": False,
}

metadata_path = (
    META_DIR
    / "04_profitability_factor_scorecard_rules.json"
)

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

# ---------- 7. 可视化 ----------
decision_colors = {
    "candidate_not_confirmed": "#54A24B",
    "not_a_standalone_factor": "#B7B7B7",
    "reject_as_standalone_factor": "#E45756",
}

plot_scorecard = scorecard.sort_values(
    "mean_rank_ic",
    ascending=True,
).copy()

colors = plot_scorecard["decision"].map(
    decision_colors
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(
    plot_scorecard["factor_label"],
    plot_scorecard["mean_rank_ic"],
    color=colors,
)
axes[0].axvline(0, color="gray", linestyle="--", linewidth=1)
axes[0].set_title("Out-of-Sample Mean RankIC")
axes[0].set_xlabel("Mean RankIC (2023-2025)")
axes[0].grid(axis="x", alpha=0.25)

axes[1].barh(
    plot_scorecard["factor_label"],
    plot_scorecard["mean_high_minus_low_return"],
    color=colors,
)
axes[1].axvline(0, color="gray", linestyle="--", linewidth=1)
axes[1].set_title("Out-of-Sample Q5 - Q1 Return")
axes[1].set_xlabel("Mean 20-Day Gross Return")
axes[1].grid(axis="x", alpha=0.25)

plt.tight_layout()

figure_path = FIG_DIR / "04_profitability_factor_scorecard.png"
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

# ---------- 8. 控制台输出 ----------
print(
    scorecard[
        [
            "factor_label",
            "mean_rank_ic",
            "rank_icir_annualized",
            "positive_rank_ic_ratio",
            "mean_high_minus_low_return",
            "high_minus_low_t_stat",
            "decision",
        ]
    ].round(4).to_string(index=False)
)

print("\n已保存：", scorecard_path)
print("已保存：", metadata_path)
print("已保存：", figure_path)

In [ ]:
# ============================================================
# 04.2 生成 GitHub README 研究摘要草稿
# ============================================================

from pathlib import Path
import pandas as pd

# ---------- 1. 路径与评分卡 ----------
PROJECT_NAME = "a-share-profitability-factor-replication"

project_candidates = [
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if path.name == PROJECT_NAME
]

if not project_candidates:
    raise RuntimeError("未找到项目根目录。")

PROJECT_DIR = project_candidates[-1]
TABLE_DIR = PROJECT_DIR / "artifacts" / "tables"
DOCS_DIR = PROJECT_DIR / "docs"
DOCS_DIR.mkdir(parents=True, exist_ok=True)

scorecard_path = (
    TABLE_DIR
    / "04_profitability_factor_scorecard.csv"
)

if not scorecard_path.exists():
    raise FileNotFoundError(
        "找不到评分卡：" + str(scorecard_path)
    )

scorecard = pd.read_csv(scorecard_path)

# ---------- 2. 生成结果 Markdown 表格 ----------
result_lines = [
    "| 因子 | 样本外 RankIC | 年化 RankICIR | 正 RankIC 比例 | Q5-Q1 单期收益 | 研究结论 |",
    "|---|---:|---:|---:|---:|---|",
]

for _, row in scorecard.iterrows():
    result_lines.append(
        "| {factor} | {rank_ic:.4f} | {rank_icir:.4f} | "
        "{positive_ratio:.2%} | {hml:.2%} | {decision} |".format(
            factor=row["factor_label"],
            rank_ic=row["mean_rank_ic"],
            rank_icir=row["rank_icir_annualized"],
            positive_ratio=row["positive_rank_ic_ratio"],
            hml=row["mean_high_minus_low_return"],
            decision=row["decision"],
        )
    )

result_table = "\n".join(result_lines)

# ---------- 3. 项目 README 内容 ----------
readme_content = f"""# A-share Profitability Factor Screening

## 项目定位

本项目是对 A 股盈利能力因子的可复现研究筛选，而不是收益展示型策略项目。

研究以招商证券《基本面量化系列研究之四：企业盈利能力评价指标的演进与优化》的思路为参考，在 RiceQuant 点时数据环境中，针对中证 800 股票池实现、检验并筛选盈利能力相关因子。

## 研究问题

盈利能力改造是否能在样本外提供稳定的横截面选股信息？

本项目比较：

1. 原始 TTM ROE；
2. 去杠杆 ROE；
3. 盈利稳定性，即过去 9 期 TTM ROE 波动率的相反数；
4. 稳定去杠杆 ROE；
5. 直接 ROIC；
6. 重构 FCFFIC。

## 研究设计

- 股票池：动态中证 800；
- 信号频率：月末最后一个交易日；
- 研究区间：2016-01 至 2025-12；
- 样本内：2016-2022；
- 样本外：2023-2025；
- 收益标签：信号日 T 后第一个交易日开盘买入、T+21 开盘卖出，持有 20 个交易日；
- 横截面处理：MAD 去极值、CITIC 2019 一级行业中性化、对数市值中性化、z-score 标准化；
- 评估指标：IC、RankIC、五分组收益、Q5-Q1 高低组收益；
- 所有个股层面的因子值、价格和收益均不保存到仓库。

## FCFFIC 重构口径

由于公开研究环境中的字段可得性限制，FCFFIC 被透明地实现为代理指标：

```text
FCFFIC = (经营活动现金流 TTM - 购建长期资产支付的现金 TTM)
         / (权益 MRQ + 有息负债 - 现金等价物)

In [ ]:
# ============================================================
# 04.2 生成 GitHub README 研究摘要草稿（兼容版）
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT_NAME = "a-share-profitability-factor-replication"

project_candidates = [
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if path.name == PROJECT_NAME
]

if not project_candidates:
    raise RuntimeError("未找到项目根目录。")

PROJECT_DIR = project_candidates[-1]
TABLE_DIR = PROJECT_DIR / "artifacts" / "tables"
DOCS_DIR = PROJECT_DIR / "docs"
DOCS_DIR.mkdir(parents=True, exist_ok=True)

scorecard_path = (
    TABLE_DIR
    / "04_profitability_factor_scorecard.csv"
)

if not scorecard_path.exists():
    raise FileNotFoundError(
        "找不到评分卡：" + str(scorecard_path)
    )

scorecard = pd.read_csv(scorecard_path)

# ---------- 1. 生成结果 Markdown 表格 ----------
result_lines = [
    "| 因子 | 样本外 RankIC | 年化 RankICIR | 正 RankIC 比例 | Q5-Q1 单期收益 | 研究结论 |",
    "|---|---:|---:|---:|---:|---|",
]

for _, row in scorecard.iterrows():
    result_lines.append(
        "| {factor} | {rank_ic:.4f} | {rank_icir:.4f} | "
        "{positive_ratio:.2%} | {hml:.2%} | {decision} |".format(
            factor=row["factor_label"],
            rank_ic=row["mean_rank_ic"],
            rank_icir=row["rank_icir_annualized"],
            positive_ratio=row["positive_rank_ic_ratio"],
            hml=row["mean_high_minus_low_return"],
            decision=row["decision"],
        )
    )

# ---------- 2. 逐行生成 README ----------
readme_lines = [
    "# A-share Profitability Factor Screening",
    "",
    "## 项目定位",
    "",
    "本项目是对 A 股盈利能力因子的可复现研究筛选，而不是收益展示型策略项目。",
    "",
    "研究以招商证券《基本面量化系列研究之四：企业盈利能力评价指标的演进与优化》的思路为参考，",
    "在 RiceQuant 点时数据环境中，针对中证 800 股票池实现、检验并筛选盈利能力相关因子。",
    "",
    "## 研究问题",
    "",
    "盈利能力改造是否能在样本外提供稳定的横截面选股信息？",
    "",
    "本项目比较：",
    "",
    "1. 原始 TTM ROE；",
    "2. 去杠杆 ROE；",
    "3. 盈利稳定性，即过去 9 期 TTM ROE 波动率的相反数；",
    "4. 稳定去杠杆 ROE；",
    "5. 直接 ROIC；",
    "6. 重构 FCFFIC。",
    "",
    "## 研究设计",
    "",
    "- 股票池：动态中证 800；",
    "- 信号频率：月末最后一个交易日；",
    "- 研究区间：2016-01 至 2025-12；",
    "- 样本内：2016-2022；",
    "- 样本外：2023-2025；",
    "- 收益标签：信号日 T 后第一个交易日开盘买入、T+21 开盘卖出，持有 20 个交易日；",
    "- 横截面处理：MAD 去极值、CITIC 2019 一级行业中性化、对数市值中性化、z-score 标准化；",
    "- 评估指标：IC、RankIC、五分组收益、Q5-Q1 高低组收益；",
    "- 所有个股层面的因子值、价格和收益均不保存到仓库。",
    "",
    "## FCFFIC 重构口径",
    "",
    "由于公开研究环境中的字段可得性限制，FCFFIC 被透明地实现为代理指标：",
    "",
    "```text",
    "FCFFIC = (经营活动现金流 TTM - 购建长期资产支付的现金 TTM)",
    "         / (权益 MRQ + 有息负债 - 现金等价物)",
    "```",
    "",
    "金融、非银金融、综合金融和房地产行业不参与 FCFFIC 测试。",
    "该实现是可复现代理指标，不声称与原研报逐点完全一致。",
    "",
    "## 样本外结果：2023-2025",
    "",
]

readme_lines.extend(result_lines)

readme_lines.extend([
    "",
    "## 研究结论",
    "",
    "1. 原始 ROE、去杠杆 ROE 与直接 ROIC 在样本外出现负向表现，不进入后续候选因子库。",
    "2. 盈利稳定性与重构 FCFFIC 在样本外保持正向 RankIC，因此保留为候选组件。",
    "3. 两个候选组件的高低组收益尚未达到显著水平，不能称为可直接交易的独立 Alpha。",
    "4. 项目不基于既有样本外结果再优化权重或拼接最优组合，以避免事后选择偏差。",
    "",
    "## 可复现性与输出",
    "",
    "`artifacts/tables/`：覆盖率、IC/RankIC、分组收益、样本外评分卡与研究决策日志。",
    "",
    "`artifacts/figures/`：券池、字段覆盖、中性化质量和因子有效性图。",
    "",
    "`artifacts/metadata/`：每一步的因子定义、样本范围、时间对齐与限制说明。",
    "",
    "## 局限与下一步",
    "",
    "- 当前不包含交易成本、涨跌停和停牌等实盘执行约束；",
    "- FCFFIC 是基于可得字段的透明代理；",
    "- 未来组合研究需要使用未参与候选筛选的新验证区间；",
    "- 下一步可加入 RONOA 的更完整金融资产字段，或在独立样本上验证盈利稳定性与 FCFFIC 的预设组合。",
])

readme_content = "\n".join(readme_lines)

# ---------- 3. 保存 ----------
readme_path = DOCS_DIR / "README_draft.md"

with open(readme_path, "w", encoding="utf-8") as file:
    file.write(readme_content)

print("已生成 README 草稿：", readme_path)
print("\n文件内容预览：\n")
print(readme_content[:1800])

In [ ]:
# ============================================================
# 04.3 GitHub 导出前文件盘点
# 只读取和生成清单，不移动、不删除任何文件
# ============================================================

from pathlib import Path

PROJECT_NAME = "a-share-profitability-factor-replication"

project_candidates = [
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if path.name == PROJECT_NAME
]

if not project_candidates:
    raise RuntimeError("未找到项目根目录。")

PROJECT_DIR = project_candidates[-1]
DOCS_DIR = PROJECT_DIR / "docs"
DOCS_DIR.mkdir(parents=True, exist_ok=True)

# 所有 Notebook：用于确认 01 和 03 的实际位置
notebook_files = sorted(PROJECT_DIR.rglob("*.ipynb"))

# 可安全开源的输出类型
safe_extensions = {
    ".ipynb",
    ".md",
    ".csv",
    ".json",
    ".png",
}

safe_files = sorted([
    path for path in PROJECT_DIR.rglob("*")
    if path.is_file()
    and path.suffix.lower() in safe_extensions
    and ".ipynb_checkpoints" not in str(path)
])

manifest_lines = [
    "# GitHub Export Manifest",
    "",
    "## Project root",
    "",
    "`{}`".format(PROJECT_DIR),
    "",
    "## Notebooks found",
    "",
]

for path in notebook_files:
    manifest_lines.append(
        "- `{}`".format(path.relative_to(PROJECT_DIR))
    )

manifest_lines.extend([
    "",
    "## Safe files for export",
    "",
])

for path in safe_files:
    manifest_lines.append(
        "- `{}`".format(path.relative_to(PROJECT_DIR))
    )

manifest_lines.extend([
    "",
    "## Do not export",
    "",
    "- Raw RiceQuant price, factor, industry, or return panels;",
    "- API credentials, tokens, cookies, or notebook checkpoints;",
    "- `tmp/`, `*.pkl`, `__pycache__/`, and any local cache files;",
    "- The accidental nested project directory after its required notebook has been copied out.",
])

manifest_path = DOCS_DIR / "EXPORT_MANIFEST.md"

with open(manifest_path, "w", encoding="utf-8") as file:
    file.write("\n".join(manifest_lines))

print("项目根目录：", PROJECT_DIR)

print("\n===== 找到的 Notebook =====")
for path in notebook_files:
    print("-", path.relative_to(PROJECT_DIR))

print("\n可安全导出的文件数：", len(safe_files))
print("已生成导出清单：", manifest_path)

In [ ]:
# ============================================================
# 04.4 安全复制 Notebook 到统一的 notebooks 目录
# 仅复制，不删除，不覆盖
# ============================================================

from pathlib import Path
import shutil
import json
from datetime import datetime

PROJECT_NAME = "a-share-profitability-factor-replication"

project_candidates = [
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if path.name == PROJECT_NAME
]

if not project_candidates:
    raise RuntimeError("未找到项目根目录。")

PROJECT_DIR = project_candidates[-1]
NOTEBOOK_DIR = PROJECT_DIR / "notebooks"
DOCS_DIR = PROJECT_DIR / "docs"

NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

copy_plan = {
    PROJECT_DIR / "00_field_audi.ipynb": (
        NOTEBOOK_DIR / "00_field_audit.ipynb"
    ),
    PROJECT_DIR
    / "a-share-profitability-factor-replication"
    / "notebooks"
    / "01_stable_roe.ipynb": (
        NOTEBOOK_DIR / "01_stable_roe.ipynb"
    ),
}

migration_records = []

for source_path, destination_path in copy_plan.items():
    if not source_path.exists():
        raise FileNotFoundError(
            "找不到源 Notebook：" + str(source_path)
        )

    if destination_path.exists():
        raise FileExistsError(
            "目标文件已存在，为避免覆盖已停止："
            + str(destination_path)
        )

    shutil.copy2(source_path, destination_path)

    if source_path.stat().st_size != destination_path.stat().st_size:
        raise RuntimeError(
            "复制后的文件大小不一致："
            + str(destination_path)
        )

    # 验证目标确实是可读取的 Jupyter Notebook
    with open(destination_path, "r", encoding="utf-8") as file:
        notebook_json = json.load(file)

    if "cells" not in notebook_json or "nbformat" not in notebook_json:
        raise RuntimeError(
            "目标文件不是有效的 Notebook："
            + str(destination_path)
        )

    migration_records.append({
        "source": str(source_path.relative_to(PROJECT_DIR)),
        "destination": str(
            destination_path.relative_to(PROJECT_DIR)
        ),
        "file_size_bytes": destination_path.stat().st_size,
        "cell_count": len(notebook_json["cells"]),
        "operation": "copy_only_source_preserved",
    })

migration_log = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "operation": (
        "Copy misplaced notebooks into the canonical notebooks directory. "
        "No source notebook was deleted or modified."
    ),
    "records": migration_records,
}

migration_log_path = DOCS_DIR / "NOTEBOOK_MIGRATION_LOG.json"

with open(migration_log_path, "w", encoding="utf-8") as file:
    json.dump(
        migration_log,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("===== 复制完成 =====")
for record in migration_records:
    print(
        "- {} -> {}".format(
            record["source"],
            record["destination"],
        )
    )
    print(
        "  文件大小：{} bytes；Notebook 单元数：{}".format(
            record["file_size_bytes"],
            record["cell_count"],
        )
    )

print("\n03 Notebook 已在正确位置：notebooks/03_roic_fcffic.ipynb")
print("迁移日志已保存：", migration_log_path)
print("\n注意：原始嵌套目录仍被保留，暂时不要删除。")

In [ ]:
# ============================================================
# 04.5 生成干净的 GitHub 导出包
# 仅复制安全文件；不删除、不修改项目原文件
# ============================================================

from pathlib import Path
import shutil
import json
import hashlib

PROJECT_NAME = "a-share-profitability-factor-replication"

project_candidates = [
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if path.name == PROJECT_NAME
]

if not project_candidates:
    raise RuntimeError("未找到项目根目录。")

PROJECT_DIR = project_candidates[-1]

EXPORT_ROOT = PROJECT_DIR / "export"
EXPORT_DIR = (
    EXPORT_ROOT
    / "a-share-profitability-factor-screening"
)

if EXPORT_DIR.exists():
    raise FileExistsError(
        "导出目录已存在。为避免覆盖，已停止："
        + str(EXPORT_DIR)
    )

# ---------- 1. 明确允许导出的 Notebook ----------
notebook_sources = [
    PROJECT_DIR / "notebooks" / "00_field_audit.ipynb",
    PROJECT_DIR / "notebooks" / "01_stable_roe.ipynb",
    PROJECT_DIR / "notebooks" / "03_roic_fcffic.ipynb",
]

for path in notebook_sources:
    if not path.exists():
        raise FileNotFoundError(
            "缺少应导出的 Notebook：" + str(path)
        )

# ---------- 2. 创建干净目录 ----------
(EXPORT_DIR / "notebooks").mkdir(
    parents=True,
    exist_ok=False,
)
(EXPORT_DIR / "artifacts" / "tables").mkdir(
    parents=True,
    exist_ok=True,
)
(EXPORT_DIR / "artifacts" / "figures").mkdir(
    parents=True,
    exist_ok=True,
)
(EXPORT_DIR / "artifacts" / "metadata").mkdir(
    parents=True,
    exist_ok=True,
)
(EXPORT_DIR / "docs").mkdir(
    parents=True,
    exist_ok=True,
)

# ---------- 3. 复制 Notebook ----------
for source_path in notebook_sources:
    shutil.copy2(
        source_path,
        EXPORT_DIR / "notebooks" / source_path.name,
    )

# ---------- 4. 复制聚合产物 ----------
artifact_rules = {
    "tables": {".csv"},
    "figures": {".png"},
    "metadata": {".json"},
}

for folder_name, allowed_extensions in artifact_rules.items():
    source_folder = (
        PROJECT_DIR / "artifacts" / folder_name
    )

    if not source_folder.exists():
        raise FileNotFoundError(
            "找不到产物目录：" + str(source_folder)
        )

    for source_path in source_folder.rglob("*"):
        if (
            source_path.is_file()
            and source_path.suffix.lower()
            in allowed_extensions
        ):
            relative_path = source_path.relative_to(
                source_folder
            )

            destination_path = (
                EXPORT_DIR
                / "artifacts"
                / folder_name
                / relative_path
            )

            destination_path.parent.mkdir(
                parents=True,
                exist_ok=True,
            )

            shutil.copy2(
                source_path,
                destination_path,
            )

# ---------- 5. README 与说明文件 ----------
readme_source = (
    PROJECT_DIR / "docs" / "README_draft.md"
)

if not readme_source.exists():
    raise FileNotFoundError(
        "找不到 README 草稿：" + str(readme_source)
    )

# GitHub 默认识别根目录 README.md
shutil.copy2(
    readme_source,
    EXPORT_DIR / "README.md",
)

migration_log_source = (
    PROJECT_DIR
    / "docs"
    / "NOTEBOOK_MIGRATION_LOG.json"
)

if migration_log_source.exists():
    shutil.copy2(
        migration_log_source,
        EXPORT_DIR
        / "docs"
        / "NOTEBOOK_MIGRATION_LOG.json",
    )

# ---------- 6. 创建 .gitignore ----------
gitignore_lines = [
    "# Jupyter",
    ".ipynb_checkpoints/",
    "__pycache__/",
    "",
    "# RiceQuant and raw vendor data caches",
    "tmp/",
    "*.pkl",
    "*.pickle",
    "*.parquet",
    "*.feather",
    "",
    "# Credentials and local configuration",
    ".env",
    "*.key",
    "*.token",
    "",
    "# System files",
    ".DS_Store",
    "Thumbs.db",
]

with open(
    EXPORT_DIR / ".gitignore",
    "w",
    encoding="utf-8",
) as file:
    file.write("\n".join(gitignore_lines) + "\n")

# ---------- 7. 创建导出说明 ----------
notice_lines = [
    "# Export Notice",
    "",
    "This package contains only notebooks, aggregate tables, figures, and metadata.",
    "",
    "It intentionally excludes raw RiceQuant factor panels, stock prices,",
    "stock-level returns, caches, credentials, and API tokens.",
    "",
    "The FCFFIC implementation is a transparent reconstructed proxy.",
    "It is not claimed to be point-identical to the source report.",
]

with open(
    EXPORT_DIR / "docs" / "EXPORT_NOTICE.md",
    "w",
    encoding="utf-8",
) as file:
    file.write("\n".join(notice_lines) + "\n")

# ---------- 8. 输出文件校验清单与 SHA256 ----------
def sha256_of_file(file_path):
    digest = hashlib.sha256()
    with open(file_path, "rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

export_files = sorted([
    path
    for path in EXPORT_DIR.rglob("*")
    if path.is_file()
])

manifest = {
    "package_name": "a-share-profitability-factor-screening",
    "file_count": len(export_files),
    "contains_raw_vendor_data": False,
    "files": [
        {
            "path": str(path.relative_to(EXPORT_DIR)),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_of_file(path),
        }
        for path in export_files
    ],
}

manifest_path = EXPORT_DIR / "EXPORT_MANIFEST.json"

with open(manifest_path, "w", encoding="utf-8") as file:
    json.dump(
        manifest,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("===== GitHub 导出包已生成 =====")
print("导出目录：", EXPORT_DIR)
print("导出文件数：", len(export_files))
print("Notebook 数：", len(notebook_sources))
print("产物目录：artifacts/tables、artifacts/figures、artifacts/metadata")
print("README：README.md")
print("校验清单：", manifest_path)
print("\n原项目文件与嵌套目录均未删除。")

In [ ]:
# ============================================================
# 04.6 将 GitHub 导出包压缩为单个 ZIP 文件
# ============================================================

from pathlib import Path
import shutil
import zipfile

PROJECT_NAME = "a-share-profitability-factor-replication"

project_candidates = [
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if path.name == PROJECT_NAME
]

if not project_candidates:
    raise RuntimeError("未找到项目根目录。")

PROJECT_DIR = project_candidates[-1]

EXPORT_DIR = (
    PROJECT_DIR
    / "export"
    / "a-share-profitability-factor-screening"
)

if not EXPORT_DIR.exists():
    raise FileNotFoundError(
        "找不到导出目录：" + str(EXPORT_DIR)
    )

ZIP_PATH = (
    PROJECT_DIR
    / "export"
    / "a-share-profitability-factor-screening.zip"
)

if ZIP_PATH.exists():
    raise FileExistsError(
        "ZIP 已存在，为避免覆盖已停止："
        + str(ZIP_PATH)
    )

# 创建 ZIP；压缩包内第一层目录就是项目名
shutil.make_archive(
    base_name=str(ZIP_PATH.with_suffix("")),
    format="zip",
    root_dir=EXPORT_DIR.parent,
    base_dir=EXPORT_DIR.name,
)

# 压缩后校验
with zipfile.ZipFile(ZIP_PATH, "r") as zip_file:
    zip_members = zip_file.namelist()

expected_prefix = (
    "a-share-profitability-factor-screening/"
)

if not all(
    member.startswith(expected_prefix)
    for member in zip_members
):
    raise RuntimeError(
        "ZIP 内目录结构异常，请勿下载。"
    )

print("ZIP 已生成：", ZIP_PATH)
print("ZIP 文件大小：{:.2f} MB".format(
    ZIP_PATH.stat().st_size / 1024 / 1024
))
print("ZIP 内文件数：", len(zip_members))
print("下载这个单独的 .zip 文件即可。")